# RiMEA 09: Crowd of People Leaving a Large Public Space

A public space with four exits and 1000 persons equally distributed in space. Free to choose between the exits. In Scenario 1 the time at wich the last person leaves the space is recorded. In Scenario 2, Exit 3 and 4 are locked and Scenario 1 is repeated in this way. The expected result is the double time compared to Scenario 1.

In [1]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 23 June 2026, 14:38 CEST


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario, run_sweep

In [3]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

##  Load and Run Scenario 1

In [4]:
SCENARIO_ZIP = Path("scenario_files") / "Rimea-09-large-public.zip"
scenario1 = load_scenario(str(SCENARIO_ZIP))
print(scenario1.summary())
result1 = run_scenario(scenario1, seed=42)
walkable1 = scenario1.walkable_polygon
df1 = result1.trajectory_dataframe()

Scenario: /Users/elif/Desktop/HiWi/jupedsim-web-community/standards/rimea/scenario_files/Rimea-09-large-public.zip
  Model:         CollisionFreeSpeedModel
  Seed:          420
  Max time:      300s
  Exits:         4
  Distributions: 1
  Stages:        0
  Zones:         0
  Journeys:      0
  Agents:        ~1000
    jps-distributions_0: 1000 agents
DEBUG: Distribution jps-distributions_0 RAW params from JSON: {'number': 1000, 'radius': 0.2, 'v0': 1.6, 'v0_std': 0.2, 'flow_start_time': 0, 'flow_end_time': 10, 'percentage': None, 'distribution_mode': 'by_number', 'use_flow_spawning': False, 'use_premovement': False, 'premovement_distribution': 'gamma', 'premovement_param_a': None, 'premovement_param_b': None, 'premovement_seed': None, 'radius_distribution': 'constant', 'v0_distribution': 'gaussian'}
DEBUG: Distribution jps-distributions_0 processed params: {'number': 1000, 'radius': 0.2, 'v0': 1.6, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement

##  Exit Time of Scenario 1

In [29]:
exit_times = (
    df1.sort_values(["id", "frame"])
    .groupby("id")
    .tail(1)[["id", "frame"]]
    .rename(columns={"frame": "exit_frame"})
)

exit_times["exit_time"] = exit_times["exit_frame"] / result1.frame_rate
print("Max exit time:", exit_times["exit_time"].max())

Max exit time: 101.2


##  Load and Run Scenario 2

In [30]:
SCENARIO_ZIP = Path("scenario_files") / "Rimea-09-large-public-locked.zip"
scenario2 = load_scenario(str(SCENARIO_ZIP))
print(scenario2.summary())
result2 = run_scenario(scenario2, seed=42)
walkable2 = scenario2.walkable_polygon
df2 = result2.trajectory_dataframe()

Scenario: /Users/elif/Desktop/HiWi/jupedsim-web-community/standards/rimea/scenario_files/Rimea-09-large-public-locked.zip
  Model:         CollisionFreeSpeedModel
  Seed:          420
  Max time:      300s
  Exits:         4
  Distributions: 1
  Stages:        0
  Zones:         0
  Journeys:      0
  Agents:        ~1000
    jps-distributions_0: 1000 agents
DEBUG: Distribution jps-distributions_0 RAW params from JSON: {'number': 1000, 'radius': 0.2, 'v0': 1.6, 'v0_std': 0.2, 'flow_start_time': 0, 'flow_end_time': 10, 'percentage': None, 'distribution_mode': 'by_number', 'use_flow_spawning': False, 'use_premovement': False, 'premovement_distribution': 'gamma', 'premovement_param_a': None, 'premovement_param_b': None, 'premovement_seed': None, 'radius_distribution': 'constant', 'v0_distribution': 'gaussian'}
DEBUG: Distribution jps-distributions_0 processed params: {'number': 1000, 'radius': 0.2, 'v0': 1.6, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_prem

##  Exit Time of Scenario 2

In [31]:
exit_times_2 = (
    df2.sort_values(["id", "frame"])
    .groupby("id")
    .tail(1)[["id", "frame"]]
    .rename(columns={"frame": "exit_frame"})
)

exit_times_2["exit_time"] = exit_times_2["exit_frame"] / result2.frame_rate
print("Max exit time with locked doors:", exit_times_2["exit_time"].max())

Max exit time with locked doors: 295.9


##  Acceptance

In [32]:
assert exit_times_2["exit_time"].max() >= 2 * exit_times["exit_time"].max(), (
    f"Expected exit_times_2 max ({exit_times_2['exit_time'].max()}) to be at least twice "
    f"exit_times max ({exit_times['exit_time'].max()})"
)
result1.cleanup()
result2.cleanup()